# Notebook 09 — Subregion PSD Exploration

Load per-pixel frequency PSD data computed by `subregion_psd_local.py` and explore the spatial and spectral structure of the E/B fields in a selected subregion of the big simulation.

In [ ]:
import numpy as np
import xarray as xr
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
from pathlib import Path

%matplotlib inline
plt.rcParams['figure.dpi'] = 120

In [ ]:
# --- Data path ---
DATA_FILE = Path('/Users/colby/Research/Projects/My_Projects/2026.Reconn_Wave_Power/big_sim/PSDs/psd_fft_x800-900_y750-1250_t0-6250.h5')

ds = xr.open_dataset(DATA_FILE, engine='h5netcdf')

# dt_output is the time between output snapshots (= dt * ndump) and is the
# correct sample spacing for the FFT frequency axis.
dt_output = ds.attrs.get('dt_output', None)
if dt_output is None:
    # fallback for files generated before this fix
    dt_output = float(ds.time[1] - ds.time[0])
    print(f"WARNING: dt_output not in attrs, inferred from time coord: {dt_output}")
else:
    print(f"dt        = {ds.attrs.get('dt', '?')}")
    print(f"ndump     = {ds.attrs.get('ndump', '?')}")
    print(f"dt_output = {dt_output}")

ds

## Variables in the file

In [ ]:
print("Data variables:")
for name, var in ds.data_vars.items():
    print(f"  {name:12s}  dims={tuple(var.dims)}  shape={var.shape}  dtype={var.dtype}")

print("\nCoordinates:")
for name, coord in ds.coords.items():
    print(f"  {name:12s}  shape={coord.shape}  [{float(coord[0].values):.4g}, {float(coord[-1].values):.4g}]")

print("\nAttributes:")
for k, v in ds.attrs.items():
    print(f"  {k}: {v}")

## Accessing the data

In [ ]:
# Access a single component — shape (x, y, frequency)
psd_bz = ds['psd_Bz']

# Coordinate arrays
freq = ds.coords['frequency'].values   # Hz
x    = ds.coords['x'].values
y    = ds.coords['y'].values

# Select nearest frequency
f_target = 0.01
psd_bz_at_f = psd_bz.sel(frequency=f_target, method='nearest')   # shape (x, y)
print(f"Nearest frequency: {float(psd_bz_at_f.frequency):.4g}")

# Spatial mean spectrum
mean_spectrum = psd_bz.mean(dim=('x', 'y'))   # shape (frequency,)

# Frequency-integrated power map
power_map = psd_bz.integrate('frequency')      # shape (x, y)

print(f"\npsd_Bz shape:      {psd_bz.shape}")
print(f"at single freq:    {psd_bz_at_f.shape}")
print(f"mean spectrum:     {mean_spectrum.shape}")
print(f"freq-integrated:   {power_map.shape}")

## Poynting Flux\n\nThe true spectral Poynting flux requires the **complex** FFT of both E and B simultaneously, so that the phase relationship between components is preserved:\n\n```\nSx(f) = Re[ Êy(f) · B̂z*(f)  −  Êz(f) · B̂y*(f) ]\n```\n\nThe auto-PSDs in this file (|FFT|²) have lost that phase information, so we load a companion file containing the complex FFT coefficients produced by `subregion_fft_local.py`.

In [ ]:
# --- Load complex FFT file (generated by subregion_fft_local.py) ---
FFT_FILE = Path('/Users/colby/Research/Projects/My_Projects/2026.Reconn_Wave_Power/big_sim/PSDs/fft_x800-900_y750-1250_t0-6250.h5')

ds_fft = xr.open_dataset(FFT_FILE, engine='h5netcdf', invalid_netcdf=True)
print(ds_fft)
print(f"\nDtypes: { {k: str(v.dtype) for k, v in ds_fft.data_vars.items()} }")

# Complex amplitude arrays — shape (x, y, frequency)
fft_Ex = ds_fft['fft_Ex']
fft_Ey = ds_fft['fft_Ey']
fft_Ez = ds_fft['fft_Ez']
fft_Bx = ds_fft['fft_Bx']
fft_By = ds_fft['fft_By']
fft_Bz = ds_fft['fft_Bz']

# True spectral Poynting flux:  S = Re(E × B*)
Sx = (fft_Ey * np.conj(fft_Bz) - fft_Ez * np.conj(fft_By)).real
Sy = (fft_Ez * np.conj(fft_Bx) - fft_Ex * np.conj(fft_Bz)).real
Sz = (fft_Ex * np.conj(fft_By) - fft_Ey * np.conj(fft_Bx)).real

print(f"\nSx shape: {Sx.shape}")
print(f"Sx range: [{float(Sx.min()):.3g}, {float(Sx.max()):.3g}]")
print(f"Sy range: [{float(Sy.min()):.3g}, {float(Sy.max()):.3g}]")
print(f"Sz range: [{float(Sz.min()):.3g}, {float(Sz.max()):.3g}]")

## 2D maps — frequency integrated

In [ ]:
comps = ['Ex', 'Ey', 'Ez', 'Bx', 'By', 'Bz']
labels = [r'$E_x$', r'$E_y$', r'$E_z$', r'$B_x$', r'$B_y$', r'$B_z$']

fig, axes = plt.subplots(2, 3, figsize=(14, 8), sharex=True, sharey=True)
for ax, comp, label in zip(axes.flat, comps, labels):
    power = ds[f'psd_{comp}'].integrate('frequency')
    im = ax.pcolormesh(x, y, power.values.T, cmap='plasma',
                       norm=mcolors.LogNorm(), shading='auto')
    plt.colorbar(im, ax=ax, label=r'$\int$ PSD $df$')
    ax.set_title(label)
    ax.set_aspect('equal')

for ax in axes[1]:
    ax.set_xlabel(r'x [$d_i$]')
for ax in axes[:, 0]:
    ax.set_ylabel(r'y [$d_i$]')

fig.suptitle('Frequency-integrated power — E & B', y=1.01)
plt.tight_layout()

## 2D maps — Y vs frequency (integrated over x)

In [ ]:
freq_nz = freq[freq > 0]   # drop DC bin (freq=0 breaks log scale)

fig, axes = plt.subplots(2, 3, figsize=(14, 8), sharex=True, sharey=True)
for ax, comp, label in zip(axes.flat, comps, labels):
    yf = ds[f'psd_{comp}'].integrate('x')                  # (y, frequency)
    yf_nz = yf.sel(frequency=yf.frequency > 0)             # drop DC
    im = ax.pcolormesh(freq_nz, y, yf_nz.values, cmap='plasma',
                       norm=mcolors.LogNorm(), shading='auto')
    plt.colorbar(im, ax=ax, label=r'$\int$ PSD $dx$')
    ax.set_title(label)
    ax.set_xscale('log')

for ax in axes[1]:
    ax.set_xlabel('Frequency')
for ax in axes[:, 0]:
    ax.set_ylabel(r'y [$d_i$]')

fig.suptitle('Y vs frequency — integrated over x', y=1.01)
plt.tight_layout()

## 2D maps — X vs frequency (integrated over y)

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(14, 8), sharex=True, sharey=True)
for ax, comp, label in zip(axes.flat, comps, labels):
    xf = ds[f'psd_{comp}'].integrate('y')                  # (x, frequency)
    xf_nz = xf.sel(frequency=xf.frequency > 0)             # drop DC
    im = ax.pcolormesh(freq_nz, x, xf_nz.values, cmap='plasma',
                       norm=mcolors.LogNorm(), shading='auto')
    plt.colorbar(im, ax=ax, label=r'$\int$ PSD $dy$')
    ax.set_title(label)
    ax.set_xscale('log')

for ax in axes[1]:
    ax.set_xlabel('Frequency')
for ax in axes[:, 0]:
    ax.set_ylabel(r'x [$d_i$]')

fig.suptitle('X vs frequency — integrated over y', y=1.01)
plt.tight_layout()

## 1D spectra — Y slices (integrated over x)\n\nEach line is a different y position; the spectrum at each y is integrated over all x.

In [ ]:
N_SLICES = 8
y_indices = np.linspace(0, len(y) - 1, N_SLICES, dtype=int)
cmap_lines = plt.cm.viridis(np.linspace(0, 1, N_SLICES))

fig, axes = plt.subplots(2, 3, figsize=(14, 8), sharex=True, sharey=True)
for ax, comp, label in zip(axes.flat, comps, labels):
    yf = ds[f'psd_{comp}'].integrate('x')
    yf_nz = yf.sel(frequency=yf.frequency > 0)
    for idx, color in zip(y_indices, cmap_lines):
        ax.loglog(freq_nz, yf_nz.isel(y=idx).values, color=color, lw=0.8)
    ax.set_title(label)

for ax in axes[1]:
    ax.set_xlabel('Frequency')
for ax in axes[:, 0]:
    ax.set_ylabel(r'$\int$ PSD $dx$')

fig.suptitle('1D spectra — Y slices (integrated over x)', y=1.01)
fig.subplots_adjust(right=0.88)
cbar_ax = fig.add_axes([0.90, 0.15, 0.02, 0.7])
sm = plt.cm.ScalarMappable(cmap='viridis',
                            norm=mcolors.Normalize(vmin=y[y_indices[0]], vmax=y[y_indices[-1]]))
fig.colorbar(sm, cax=cbar_ax, label=r'y [$d_i$]')

## 1D spectra — X slices (integrated over y)\n\nEach line is a different x position; the spectrum at each x is integrated over all y.

In [ ]:
x_indices = np.linspace(0, len(x) - 1, N_SLICES, dtype=int)
cmap_lines = plt.cm.plasma(np.linspace(0, 1, N_SLICES))

fig, axes = plt.subplots(2, 3, figsize=(14, 8), sharex=True, sharey=True)
for ax, comp, label in zip(axes.flat, comps, labels):
    xf = ds[f'psd_{comp}'].integrate('y')
    xf_nz = xf.sel(frequency=xf.frequency > 0)
    for idx, color in zip(x_indices, cmap_lines):
        ax.loglog(freq_nz, xf_nz.isel(x=idx).values, color=color, lw=0.8)
    ax.set_title(label)

for ax in axes[1]:
    ax.set_xlabel('Frequency')
for ax in axes[:, 0]:
    ax.set_ylabel(r'$\int$ PSD $dy$')

sm = plt.cm.ScalarMappable(cmap='plasma',
                            norm=mcolors.Normalize(vmin=x[x_indices[0]], vmax=x[x_indices[-1]]))
fig.colorbar(sm, ax=axes, label=r'x [$d_i$]', shrink=0.6)
fig.suptitle('1D spectra — X slices (integrated over y)', y=1.01)
plt.tight_layout()

---\n## Poynting flux — Sx only\n\nAll four views repeated for the x-component of the Poynting flux.\n`RdBu_r` colormap since Sx is signed.

In [ ]:
# Helper: symmetric colormap norm centred on zero
def sym_norm(data):
    vmax = float(np.abs(data).max())
    return mcolors.Normalize(vmin=-vmax, vmax=vmax)

# --- 2D: frequency integrated ---
sx_int = Sx.integrate('frequency')   # (x, y)
fig, ax = plt.subplots(figsize=(7, 5))
im = ax.pcolormesh(x, y, sx_int.values.T, cmap='RdBu_r',
                   norm=sym_norm(sx_int.values), shading='auto')
plt.colorbar(im, ax=ax, label=r'$\int S_x \, df$')
ax.set_xlabel(r'x [$d_i$]');  ax.set_ylabel(r'y [$d_i$]')
ax.set_title(r'$S_x$ — frequency integrated')
ax.set_aspect('equal')
plt.tight_layout()

In [ ]:
# --- 2D: Y vs frequency (integrate over x) ---
sx_yf = Sx.integrate('x')
sx_yf_nz = sx_yf.sel(frequency=sx_yf.frequency > 0)
fig, ax = plt.subplots(figsize=(7, 5))
im = ax.pcolormesh(freq_nz, y, sx_yf_nz.values, cmap='RdBu_r',
                   norm=sym_norm(sx_yf_nz.values), shading='auto')
plt.colorbar(im, ax=ax, label=r'$\int S_x \, dx$')
ax.set_xscale('log')
ax.set_xlabel('Frequency');  ax.set_ylabel(r'y [$d_i$]')
ax.set_title(r'$S_x$ — Y vs frequency')
plt.tight_layout()

In [ ]:
# --- 2D: X vs frequency (integrate over y) ---
sx_xf = Sx.integrate('y')
sx_xf_nz = sx_xf.sel(frequency=sx_xf.frequency > 0)
fig, ax = plt.subplots(figsize=(7, 5))
im = ax.pcolormesh(freq_nz, x, sx_xf_nz.values, cmap='RdBu_r',
                   norm=sym_norm(sx_xf_nz.values), shading='auto')
plt.colorbar(im, ax=ax, label=r'$\int S_x \, dy$')
ax.set_xscale('log')
ax.set_xlabel('Frequency');  ax.set_ylabel(r'x [$d_i$]')
ax.set_title(r'$S_x$ — X vs frequency')
plt.tight_layout()

In [ ]:
# --- 1D: Y slices (integrated over x) ---
cmap_lines = plt.cm.viridis(np.linspace(0, 1, N_SLICES))
fig, ax = plt.subplots(figsize=(7, 5))
for idx, color in zip(y_indices, cmap_lines):
    ax.plot(freq_nz, sx_yf_nz.isel(y=idx).values, color=color, lw=0.9)
ax.set_xscale('log')
ax.set_xlabel('Frequency');  ax.set_ylabel(r'$\int S_x \, dx$')
ax.set_title(r'$S_x$ — Y slices')
ax.axhline(0, color='k', lw=0.5, ls='--')
sm = plt.cm.ScalarMappable(cmap='viridis',
                            norm=mcolors.Normalize(vmin=y[y_indices[0]], vmax=y[y_indices[-1]]))
plt.colorbar(sm, ax=ax, label=r'y [$d_i$]')
plt.tight_layout()

In [ ]:
# --- 1D: Y slices (integrated over x) ---
cmap_lines = plt.cm.viridis(np.linspace(0, 1, N_SLICES))
fig, ax = plt.subplots(figsize=(7, 5))
for idx, color in zip(y_indices, cmap_lines):
    ax.loglog(freq_nz, np.abs(sx_yf_nz.isel(y=idx)).values, color=color, lw=0.9)
ax.set_xscale('log')
ax.set_xlabel('Frequency');  ax.set_ylabel(r'$\int S_x \, dx$')
ax.set_title(r'$S_x$ — Y slices')
ax.axhline(0, color='k', lw=0.5, ls='--')
sm = plt.cm.ScalarMappable(cmap='viridis',
                            norm=mcolors.Normalize(vmin=y[y_indices[0]], vmax=y[y_indices[-1]]))
plt.colorbar(sm, ax=ax, label=r'y [$d_i$]')
plt.tight_layout()

In [ ]:
# --- 1D: X slices (integrated over y) ---
cmap_lines = plt.cm.plasma(np.linspace(0, 1, N_SLICES))
fig, ax = plt.subplots(figsize=(7, 5))
for idx, color in zip(x_indices, cmap_lines):
    ax.plot(freq_nz, sx_xf_nz.isel(x=idx).values, color=color, lw=0.9)
ax.set_xscale('log')
ax.set_xlabel('Frequency');  ax.set_ylabel(r'$\int S_x \, dy$')
ax.set_title(r'$S_x$ — X slices')
ax.axhline(0, color='k', lw=0.5, ls='--')
sm = plt.cm.ScalarMappable(cmap='plasma',
                            norm=mcolors.Normalize(vmin=x[x_indices[0]], vmax=x[x_indices[-1]]))
plt.colorbar(sm, ax=ax, label=r'x [$d_i$]')
plt.tight_layout()